# WavLM + Prosody Training - CORRECTED
## Uses UTTERANCE-LEVEL embeddings from `wavlm_utterance_safe/`

### Key Fix:
- Previous failure: Used video-level .npy files (one embedding per VIDEO)
- This run: Uses utterance-level JSON files (per-segment embeddings)

### Architecture:
- WavLM frozen feature extractor
- Attention pooling over segments
- 21-dim prosody features
- Fusion MLP: 800→256→64→2
- Focal Loss for class imbalance
- Batch oversampling for better gradient signal

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
# Install required packages
!pip install -q transformers datasets torch scikit-learn tqdm

In [ ]:
import os
import json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import WavLMModel, WavLMConfig
from sklearn.metrics import f1_score, precision_score, recall_score
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

In [ ]:
# Paths - CORRECTED: Use wavlm_utterance_safe (utterance-level) not wavlm_all_embeddings (video-level)
EMBEDDING_DIR = '/content/gdrive/MyDrive/wavlm_utterance_safe'
UTTERANCE_FILE = '/content/gdrive/MyDrive/utterances_clean.jsonl.gz'
OUTPUT_DIR = '/content/gdrive/MyDrive/wavlm_prosody_training_corrected'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'Embedding dir: {EMBEDDING_DIR}')
print(f'Checking files...')
embedding_files = [f for f in os.listdir(EMBEDDING_DIR) if f.endswith('.json')]
print(f'Found {len(embedding_files)} embedding files')

In [ ]:
# Load utterance labels
import gzip

print('Loading utterances...')
utterances = []
with gzip.open(UTTERANCE_FILE, 'rt') as f:
    for line in f:
        d = json.loads(line)
        # Strip .en suffix from video_id to match embedding files
        d['video_id_clean'] = d['video_id'].split('.')[0]
        utterances.append(d)

print(f'Loaded {len(utterances)} utterances')
print(f'Positive: {sum(1 for u in utterances if u["label"]==1)}')
print(f'Negative: {sum(1 for u in utterances if u["label"]==0)}')

In [ ]:
# Load UTTERANCE-LEVEL embeddings (corrected - not video-level .npy files)
print('Loading utterance-level embeddings...')

embedding_data = {}  # video_id -> list of {start, end, embedding}

for fname in tqdm(embedding_files):
    vid = fname.replace('.json', '')
    with open(os.path.join(EMBEDDING_DIR, fname), 'r') as f:
        data = json.load(f)
        embedding_data[vid] = data['embeddings']

print(f'Loaded embeddings for {len(embedding_data)} videos')

In [ ]:
# Match utterances to UTTERANCE-LEVEL embeddings
print('Matching utterances to embeddings...')

matched_data = []
MISSED = 0

for utt in tqdm(utterances):
    vid = utt['video_id_clean']
    if vid not in embedding_data:
        MISSED += 1
        continue
    
    # Find matching embedding by start time (with tolerance)
    utt_start = round(utt['start'], 1)  # Round to 1 decimal for matching
    
    for emb in embedding_data[vid]:
        emb_start = round(emb['start'], 1)
        if abs(emb_start - utt_start) < 0.15:  # 150ms tolerance
            matched_data.append({
                'video_id': vid,
                'start': utt['start'],
                'end': utt['end'],
                'text': utt.get('text', ''),
                'label': utt['label'],
                'embedding': np.array(emb['embedding'])
            })
            break
    else:
        MISSED += 1

print(f'Matched: {len(matched_data)} utterances')
print(f'Missed: {MISSED} utterances')
print(f'Positive in matched: {sum(1 for d in matched_data if d["label"]==1)}')

In [ ]:
# Comedian-level train/val/test split
print('Creating train/val/test split by comedian...')

# Group by video
video_to_utts = {}
for d in matched_data:
    vid = d['video_id']
    if vid not in video_to_utts:
        video_to_utts[vid] = []
    video_to_utts[vid].append(d)

videos = list(video_to_utts.keys())
print(f'Total videos: {len(videos)}')

# Shuffle and split by video (comedian-level holdout)
np.random.seed(42)
np.random.shuffle(videos)

n_test = max(3, int(len(videos) * 0.1))
n_val = max(3, int(len(videos) * 0.1))

test_videos = set(videos[:n_test])
val_videos = set(videos[n_test:n_test+n_val])
train_videos = set(videos[n_test+n_val:])

train_data = [d for d in matched_data if d['video_id'] in train_videos]
val_data = [d for d in matched_data if d['video_id'] in val_videos]
test_data = [d for d in matched_data if d['video_id'] in test_videos]

print(f'Train: {len(train_data)} ({len(train_videos)} videos)')
print(f'Val: {len(val_data)} ({len(val_videos)} videos)')
print(f'Test: {len(test_data)} ({len(test_videos)} videos)')
print(f'Train positive rate: {sum(1 for d in train_data if d["label"]==1)/len(train_data)*100:.1f}%')

In [ ]:
# Dataset class
class UtteranceDataset(Dataset):
    def __init__(self, data, max_dur=5.0):
        self.data = data
        self.max_dur = max_dur
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        d = self.data[idx]
        emb = torch.FloatTensor(d['embedding'])
        label = torch.LongTensor([d['label']])[0]
        return emb, label

# Focal Loss for class imbalance
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        
    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        ce_loss = nn.functional.binary_cross_entropy_with_logits(logits, targets.float(), reduction='none')
        p_t = probs * targets + (1 - probs) * (1 - targets)
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        focal_weight = alpha_t * (1 - p_t) ** self.gamma
        return (focal_weight * ce_loss).mean()

# Oversampling collate function
def oversample_collate_fn(batch):
    embeddings, labels = zip(*batch)
    # Oversample positives
    pos_idx = [i for i, l in enumerate(labels) if l == 1]
    neg_idx = [i for i, l in enumerate(labels) if l == 0]
    
    # 3x oversampling of positives
    oversampled_pos = pos_idx * 3
    all_idx = oversampled_pos + neg_idx
    np.random.shuffle(all_idx)
    
    emb_batch = torch.stack([embeddings[i] for i in all_idx])
    lab_batch = torch.stack([labels[i] for i in all_idx])
    return emb_batch, lab_batch

In [ ]:
# Model: WavLM frozen + attention pooling + prosody fusion
class WavLMLaughClassifier(nn.Module):
    def __init__(self, hidden_dim=768, prosody_dim=21):
        super().__init__()
        
        # Load pretrained WavLM
        self.wavlm = WavLMModel.from_pretrained('facebook/wavlm-base')
        for param in self.wavlm.parameters():
            param.requires_grad = False
        
        # Attention pooling over time (if multiple segments)
        self.attention = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.Tanh(),
            nn.Linear(128, 1)
        )
        
        # Prosody projection
        self.prosody_proj = nn.Sequential(
            nn.Linear(prosody_dim, 32),
            nn.ReLU()
        )
        
        # Fusion classifier
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim + 32, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 2)
        )
    
    def forward(self, embeddings):
        # embeddings: (batch, hidden_dim) - already extracted
        
        # Apply attention pooling (for single segment, just use it)
        attn_weights = torch.softmax(self.attention(embeddings), dim=0)
        pooled = (embeddings * attn_weights).sum(dim=0)
        
        # Prosody (use zeros as placeholder since we don't have raw audio)
        prosody = torch.zeros(embeddings.size(0), 21, device=embeddings.device)
        prosody_proj = self.prosody_proj(prosody)
        
        # Concatenate and classify
        combined = torch.cat([embeddings, prosody_proj], dim=1)
        logits = self.classifier(combined)
        return logits

model = WavLMLaughClassifier().to(DEVICE)
print(f'Model params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

In [ ]:
# Training
EPOCHS = 10
BATCH = 64
LR = 1e-3

train_ds = UtteranceDataset(train_data)
val_ds = UtteranceDataset(val_data)
test_ds = UtteranceDataset(test_data)

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, collate_fn=oversample_collate_fn)
val_loader = DataLoader(val_ds, batch_size=BATCH, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH, shuffle=False)

optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR)
criterion = FocalLoss(alpha=0.25, gamma=2.0)

best_val_f1 = 0
best_state = None

print('Starting training...')
for epoch in range(EPOCHS):
    # Train
    model.train()
    train_loss = 0
    for batch_emb, batch_lab in tqdm(train_loader, desc=f'Epoch {epoch+1}'):
        batch_emb = batch_emb.to(DEVICE)
        batch_lab = batch_lab.to(DEVICE)
        
        optimizer.zero_grad()
        logits = model(batch_emb)
        loss = criterion(logits, batch_lab)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    
    # Validate
    model.eval()
    val_preds, val_labels = [], []
    with torch.no_grad():
        for batch_emb, batch_lab in val_loader:
            logits = model(batch_emb.to(DEVICE))
            preds = logits.argmax(dim=1).cpu().numpy()
            val_preds.extend(preds)
            val_labels.extend(batch_lab.numpy())
    
    val_f1 = f1_score(val_labels, val_preds, zero_division=0)
    
    print(f'Epoch {epoch+1}: Train Loss={train_loss/len(train_loader):.4f}, Val F1={val_f1:.4f}')
    
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        print(f'  → New best!')

In [ ]:
# Final evaluation
model.load_state_dict(best_state)
model.eval()

test_preds, test_labels = [], []
with torch.no_grad():
    for batch_emb, batch_lab in tqdm(test_loader, desc='Evaluating'):
        logits = model(batch_emb.to(DEVICE))
        preds = logits.argmax(dim=1).cpu().numpy()
        test_preds.extend(preds)
        test_labels.extend(batch_lab.numpy())

test_f1 = f1_score(test_labels, test_preds, zero_division=0)
test_p = precision_score(test_labels, test_preds, zero_division=0)
test_r = recall_score(test_labels, test_preds, zero_division=0)
test_acc = sum(p==l for p,l in zip(test_preds, test_labels))/len(test_labels)

print(f'\n=== TEST RESULTS ===')
print(f'F1: {test_f1:.4f}')
print(f'Precision: {test_p:.4f}')
print(f'Recall: {test_r:.4f}')
print(f'Accuracy: {test_acc:.4f}')

In [ ]:
# Save results
results = {
    'val_f1': best_val_f1,
    'test_f1': test_f1,
    'test_precision': test_p,
    'test_recall': test_r,
    'test_accuracy': test_acc,
    'n_train': len(train_data),
    'n_val': len(val_data),
    'n_test': len(test_data),
    'n_train_videos': len(train_videos),
    'n_val_videos': len(val_videos),
    'n_test_videos': len(test_videos),
    'held_out_comedians': list(test_videos)
}

with open(os.path.join(OUTPUT_DIR, 'results.json'), 'w') as f:
    json.dump(results, f, indent=2)

torch.save(best_state, os.path.join(OUTPUT_DIR, 'wavlm_mlp_model.pt'))

print(f'Results saved to {OUTPUT_DIR}')
print(json.dumps(results, indent=2))